### AGENT EVALUATION

In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [6]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 4.0, "section": 0.2},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [7]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [8]:
rec = ground_truth[0]
rec
result = runner.loop(prompt=rec["question"])

In [9]:
rec

{'question': 'Can I still join the course if I found it late?',
 'document': '74eb249bbf'}

In [10]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Can I still join the course if I found it late?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"join course late found it late enrollment late join"}', call_id='call_DjvimrQT1kIrFnOiID2NZeb2', name='search', type='function_call', id='fc_06614a15a70e3ad3006a5aa85c067c8191883a37f9a86c7774', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_DjvimrQT1kIrFnOiID2NZeb2',
  'output': '[\n  {\n    "id": "cdc3b285e5",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Can I submit homework after the deadline, or get a deadline extension?",\n    "answer": "No. We don\'t give individual deadline extensions, and once the homework submission form 

In [11]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [12]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"join course late found it late enrollment late join"}'}]

In [13]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [15]:
answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [14]:
result.last_message

'Yes — you can still join even if you found the course late. The FAQ says: “You can start whenever you want.”\n\nOne important note: if you want a certificate, you need to submit your project while submissions are still open.'

In [16]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Can I still join the course if I found it late?',
 'answer_agent': 'Yes — you can still join even if you found the course late. The FAQ says: “You can start whenever you want.”\n\nOne important note: if you want a certificate, you need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"join course late found it late enrollment late join"}'}],
 'cost': Decimal('0.00105825'),
 'document': '74eb249bbf'}

In [17]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [18]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [19]:
df_agent = pd.DataFrame(agent_answers)

In [20]:
df_agent["cost"].sum()

Decimal('0.06700425')

In [21]:
df_agent.head(3)

,question,answer_agent,answer_orig,tool_calls,cost,document
0,Can I still join the course if I found it late?,Yes — you can still join the course even if yo...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.00112575,74eb249bbf
1,Am I allowed to enroll after the course has al...,Yes — you can start the course whenever you wa...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""en...",0.0011550,74eb249bbf
2,"If I join now, can I still get a certificate?","Yes — you can still join now, but to get a cer...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""jo...",0.0011775,74eb249bbf


In [22]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [23]:
df_agent_loaded = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent_loaded.to_dict(orient="records")

#### Judging answers and trajectories
A good trajectory is not just "many tool calls". A good trajectory uses the available tools in a way that helps answer the question.

For our search agent, a good trajectory has these properties:

- The search query is relevant to the user question
- The query includes the important keywords from the question
- The agent avoids duplicate searches with the same arguments
- If it searches more than once, the next query is a useful refinement
- It usually uses 1 search call
- 2-3 calls can be okay for harder questions
- More than 3 search calls needs a clear reason
- The tool calls support the final answer
- The agent does not stop too early or keep searching without a reason

In [24]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [25]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [42]:
import json
import ast
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        ## tool_calls = json.loads(tool_calls)
        try:
            tool_calls = json.loads(tool_calls)
        except json.JSONDecodeError:
            # If JSON parsing fails, try to evaluate as Python literal
            
            try:
                tool_calls = ast.literal_eval(tool_calls)
            except (ValueError, SyntaxError):
                tool_calls = []

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [43]:
rec = agent_answers[0]
rec

{'question': 'Can I still join the course if I found it late?',
 'answer_agent': 'Yes — you can still join the course even if you found it late. \n\nAccording to the FAQ, you can start whenever you want, and the videos and GitHub materials are available. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{\'name\': \'search\', \'arguments\': \'{"query":"late join course enroll after start join late found it late FAQ"}\'}]',
 'cost': 0.00112575,
 'document': '74eb249bbf'}

In [44]:
tool_calls = rec["tool_calls"]
tool_calls

'[{\'name\': \'search\', \'arguments\': \'{"query":"late join course enroll after start join late found it late FAQ"}\'}]'

In [45]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It states that you can still join the course if you found it late, and it correctly adds that to receive a certificate you must submit your project while submissions are still being accepted. The extra detail about starting whenever you want and materials being available is consistent with the FAQ-style answer and does not change the meaning.', answer_score='good', trajectory_reasoning='The single search query was relevant to the user’s question and included key ideas like joining late and FAQ. One search call was sufficient for this simple question, and there were no unnecessary duplicate calls. The tool use reasonably supported the final answer.', trajectory_score='good')

In [46]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [47]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [48]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [49]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [50]:
calc_total_price(usages)

0.05432699999999999

In [51]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    47
bad      3
Name: count, dtype: int64

In [52]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [53]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)